# Lot D - Analyse quantitative et qualitative

Ce notebook est la version d'analyse pour le Lot D du projet ELOQUENT.

Objectif : comparer les modeles, les variantes, les langues et les types de dataset, puis completer les resultats quantitatifs par une analyse qualitative d'exemples.

Exigences du Lot D couvertes par ce notebook :

- statistiques simples : longueur, reponses vides, respect de la concision ;
- mesure semantique par embeddings pour la coherence sur `specific` et la diversite sur `unspecific` ;
- comparaison structuree entre baseline et variante lorsque les runs sont disponibles ;
- selection d'exemples qualitatifs : genericite, stereotypes, hallucinations culturelles possibles, non-respect de consigne, cas incoherents ou tres variables.

Le notebook est concu pour rester executable meme si certains runs ne sont pas encore termines.

## 1. Configuration et dependances

La configuration importante est regroupee ici. Les langues du projet sont `fr`, `it`, `de`, `es`, `en`.

In [ ]:
import hashlib
import importlib.util
import json
import math
import re
import unicodedata
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

TARGET_LANGUAGES = ["fr", "it", "de", "es", "en"]
DATASET_TYPES = ["specific", "unspecific"]

EXPECTED_PER_LANGUAGE = {
    "specific": 4000,
    "unspecific": 100,
}

BASELINE_STRATEGY = "vanilla"
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
RUNS_DIR = None
CACHE_DIR = None

REQUIRED_MODULES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "sentence_transformers": "sentence-transformers",
}

missing = [pip_name for module, pip_name in REQUIRED_MODULES.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Dependances manquantes pour l'analyse complete :")
    print("%pip install " + " ".join(missing))
else:
    print("Dependances principales disponibles.")

In [ ]:
def find_project_root():
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "output" / "runs").exists():
            return candidate
    return here


PROJECT_ROOT = find_project_root()
RUNS_DIR = PROJECT_ROOT / "data" / "output" / "runs"
CACHE_DIR = PROJECT_ROOT / "data" / "output" / "analysis_cache" / "embeddings"

print(f"Racine projet : {PROJECT_ROOT.resolve()}")
print(f"Dossier runs  : {RUNS_DIR.resolve()}")


## 2. Audit des runs disponibles

Cette section verifie ce qui est disponible avant d'analyser. Elle distingue les donnees attendues, partielles, completes et les fichiers de langues hors protocole.

In [ ]:
OUTPUT_RE = re.compile(r"^(?P<lang>[a-z]{2})_(?P<dataset>specific|unspecific)_output\.jsonl$")


def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def read_yaml_config(path):
    """Lit config_snapshot.yaml (sauvegarde par le pipeline) en fallback.
    Utile quand run_metadata.json est absent (run interrompu : quota, timeout)."""
    if not path.exists():
        return {}
    try:
        import yaml
        return yaml.safe_load(path.read_text(encoding="utf-8")) or {}
    except Exception:
        return {}


def count_jsonl(path):
    if not path or not Path(path).exists():
        return 0
    with Path(path).open("r", encoding="utf-8") as handle:
        return sum(1 for line in handle if line.strip())


def normalize_strategy(value, run_name):
    raw = str(value or "").lower().strip()
    name = run_name.lower()
    if raw in {"vanilla", "baseline"} or "vanilla" in name:
        return "vanilla"
    if raw in {"tuned", "tuning", "tune"} or any(token in name for token in ["tuned", "tuning", "tune"]):
        return "tuned"
    if raw in {"rewrite", "rewriting", "reformulation"} or "rewrite" in name:
        return "rewrite"
    if raw in {"prefix_suffix", "prefix", "suffix"} or "prefix" in name or "suffix" in name:
        return "prefix_suffix"
    if raw in {"system_prompt", "prompting"} or "system" in name:
        return "system_prompt"
    return raw or "unknown"


def infer_provider(metadata, run_name):
    provider = str(metadata.get("provider") or "").strip()
    if provider:
        return provider
    name = run_name.lower()
    if "groq" in name:
        return "groq"
    if "qwen" in name:
        return "qwen_ollama"
    if "ollama" in name:
        return "ollama"
    return "unknown"


def build_run_catalog(runs_dir=RUNS_DIR):
    rows = []
    unexpected_rows = []
    if not runs_dir.exists():
        return pd.DataFrame(rows), pd.DataFrame(unexpected_rows)

    for run_dir in sorted(path for path in runs_dir.iterdir() if path.is_dir()):
        metadata = read_json(run_dir / "run_metadata.json")
        # Fallback config_snapshot.yaml si metadata absent/incomplet (run interrompu).
        config = read_yaml_config(run_dir / "config_snapshot.yaml")
        run_name = run_dir.name
        run_id = metadata.get("run_id") or config.get("run_id") or run_name
        provider = infer_provider(metadata, run_name)
        if provider == "unknown":
            provider = infer_provider(config, run_name)
        model = (metadata.get("model") or metadata.get("model_name")
                 or config.get("model") or "unknown")
        strategy_raw = (metadata.get("strategy")
                        or (config.get("prompting") or {}).get("strategy"))
        strategy = normalize_strategy(strategy_raw, run_name)
        output_files = {}

        for file_path in run_dir.glob("*_output.jsonl"):
            match = OUTPUT_RE.match(file_path.name)
            if not match:
                continue
            lang = match.group("lang")
            dataset_type = match.group("dataset")
            output_files[(lang, dataset_type)] = file_path
            if lang not in TARGET_LANGUAGES:
                unexpected_rows.append({
                    "run_name": run_name,
                    "language": lang,
                    "dataset_type": dataset_type,
                    "file": file_path.name,
                    "reason": "langue hors TARGET_LANGUAGES, ignoree par l'analyse",
                })

        for lang in TARGET_LANGUAGES:
            for dataset_type in DATASET_TYPES:
                file_path = output_files.get((lang, dataset_type))
                n_responses = count_jsonl(file_path)
                expected = EXPECTED_PER_LANGUAGE[dataset_type]
                if n_responses == 0:
                    status = "missing"
                elif n_responses < expected:
                    status = "partial"
                else:
                    status = "complete"
                rows.append({
                    "run_dir": str(run_dir),
                    "run_name": run_name,
                    "run_id": run_id,
                    "provider": provider,
                    "model": model,
                    "strategy": strategy,
                    "dataset_type": dataset_type,
                    "language": lang,
                    "file_path": str(file_path) if file_path else None,
                    "n_responses": n_responses,
                    "expected": expected,
                    "coverage_pct": round(n_responses / expected * 100, 1),
                    "status": status,
                })
    return pd.DataFrame(rows), pd.DataFrame(unexpected_rows)


run_catalog, unexpected_files = build_run_catalog()

if run_catalog.empty:
    print("Aucun run detecte.")
else:
    display(run_catalog.sort_values(["strategy", "provider", "dataset_type", "language", "run_name"]))

if not unexpected_files.empty:
    print("Fichiers detectes mais ignores car hors langues cible :")
    display(unexpected_files)


In [ ]:
def summarize_coverage(catalog):
    if catalog.empty:
        return pd.DataFrame()
    return (
        catalog.groupby(["provider", "model", "strategy", "dataset_type", "language"], dropna=False)
        .agg(
            responses=("n_responses", "sum"),
            expected=("expected", "max"),
            files=("file_path", lambda values: values.notna().sum()),
        )
        .reset_index()
        .assign(
            coverage_pct=lambda df: (df["responses"] / df["expected"] * 100).round(1),
            status=lambda df: np.select(
                [df["responses"].eq(0), df["responses"].lt(df["expected"]), df["responses"].ge(df["expected"])],
                ["missing", "partial", "complete"],
                default="unknown",
            ),
        )
    )


coverage = summarize_coverage(run_catalog)
display(coverage.sort_values(["dataset_type", "strategy", "provider", "language"]))

if not coverage.empty:
    fig, ax = plt.subplots(figsize=(12, 4))
    plot_df = coverage.copy()
    plot_df["condition"] = plot_df["provider"] + " / " + plot_df["strategy"] + " / " + plot_df["dataset_type"]
    sns.barplot(data=plot_df, x="language", y="coverage_pct", hue="condition", ax=ax)
    ax.axhline(100, color="black", linestyle="--", linewidth=1)
    ax.set_title("Completude des runs par langue")
    ax.set_xlabel("Langue")
    ax.set_ylabel("% de l'objectif")
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()

## 3. Chargement normalise des reponses

Toutes les reponses sont converties dans un seul tableau `df_all`. Les analyses suivantes utilisent ce tableau unique.

In [ ]:
def normalize_record(record):
    row = dict(record)
    for prompt_col in ["prompt", "question", "query", "text"]:
        if prompt_col in row:
            row["prompt"] = row.get(prompt_col) or ""
            break
    row.setdefault("prompt", "")
    row.setdefault("answer", "")
    row.setdefault("id", "unknown")
    row["prompt"] = str(row["prompt"])
    row["answer"] = "" if row["answer"] is None else str(row["answer"])
    row["id"] = str(row["id"])
    return row


def read_jsonl(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(normalize_record(json.loads(line)))
    return records


def load_responses(catalog):
    rows = []
    available = catalog[catalog["n_responses"] > 0].copy()
    for _, meta in available.iterrows():
        for row in read_jsonl(meta["file_path"]):
            row.update({
                "run_name": meta["run_name"],
                "run_id": meta["run_id"],
                "provider": meta["provider"],
                "model": meta["model"],
                "strategy": meta["strategy"],
                "dataset_type": meta["dataset_type"],
                "language": meta["language"],
            })
            rows.append(row)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    df["question_id"] = df["id"].str.split("-").str[0]
    df["culture_id"] = np.where(df["id"].str.contains("-"), df["id"].str.split("-").str[1], "implicit")
    df["run_label"] = df["provider"] + " / " + df["model"] + " / " + df["strategy"] + " / " + df["dataset_type"]
    return df


df_all = load_responses(run_catalog)
print(f"Reponses chargees : {len(df_all)}")
if not df_all.empty:
    display(df_all[["run_label", "language", "id", "prompt", "answer"]].head())

## 4. Metriques textuelles simples

Ces metriques repondent a la premiere demande quantitative : longueur, taux de reponses vides, concision et respect de consigne.

La detection de pays n'utilise plus une liste manuelle de langues/pays. Elle cherche d'abord le pays explicitement present dans le prompt `specific`, puis verifie s'il reapparait dans la reponse.

In [ ]:
# Extraction du PAYS de contexte (questions "specific"). Un pattern PAR LANGUE :
# le routage par langue evite qu'un pattern (ex. FR, qui partage la preposition
# "en"/"a") capture par erreur une phrase d'une autre langue. Le VERBE
# d'habitation est insensible a la casse via (?i:...) car il peut debuter la
# phrase ("Viviamo", "Wir"). Le groupe PAYS garde sa casse pour exiger une
# initiale majuscule. La capture s'arrete sur la ponctuation, la fin de chaine,
# une conjonction de coordination suivie d'un mot en MINUSCULE (ce qui preserve
# "Bosnia y Herzegovina"), ou le mot-instruction du template ("Repondez"...) qui
# sert de filet quand la donnee source est malformee (point manquant).
_INSTR = {
    "fr": r"R[ée]pond",
    "en": r"Answer",
    "es": r"Responde",
    "it": r"Rispond",
    "de": r"Antworten",
}
PATTERN_BY_LANG = {
    "fr": r"(?i:habit\w*|viv\w*|vis)\s+(?:\w+\s+)*?(?i:en|au|aux|à|a)\s+(?:les\s+)?"
          r"([A-ZÀ-Ÿ][^\.,;!?]*?)(?=\s*(?:[\.,;!?]|\bet\s+[a-zà-ÿ]|\b" + _INSTR["fr"] + r"|$))",
    "en": r"\b(?i:liv\w*)\s+(?:\w+\s+)*?in\s+(?:the\s+)?"
          r"([A-Z][^\.,;!?]*?)(?=\s*(?:[\.,;!?]|\band\s+[a-z]|\b" + _INSTR["en"] + r"|$))",
    "es": r"\b(?i:viv\w*)\s+(?:\w+\s+)*?en\s+(?:la\s+|el\s+|los\s+|las\s+)?"
          r"([A-ZÁÉÍÓÚÑ][^\.,;!?]*?)(?=\s*(?:[\.,;!?]|\by\s+[a-zá-ú]|\b" + _INSTR["es"] + r"|$))",
    "it": r"\b(?i:viv\w*|abit\w*)\s+(?:\w+\s+)*?(?:in|a)\s+(?:il\s+|la\s+|le\s+|gli\s+|nei\s+|nella\s+|nelle\s+)?"
          r"([A-ZÀ-Ù][^\.,;!?]*?)(?=\s*(?:[\.,;!?]|\be\s+[a-zà-ù]|\b" + _INSTR["it"] + r"|$))",
    "de": r"\b(?i:leb\w*|wohn\w*)\s+(?:\w+\s+)*?in\s+(?:den\s+|der\s+|dem\s+|das\s+|die\s+)?"
          r"([A-ZÄÖÜ][^\.,;!?]*?)(?=\s*(?:[\.,;!?]|\bund\s+[a-zä-ü]|\b" + _INSTR["de"] + r"|$))",
}
_COMPILED_BY_LANG = {lang: re.compile(p) for lang, p in PATTERN_BY_LANG.items()}
# Ordre de repli deterministe quand la langue est inconnue.
_FALLBACK_ORDER = ["en", "de", "it", "es", "fr"]

# Heuristiques LEXICALES (mots-cles). Indicatives, NON semantiques :
# elles servent uniquement a pre-selectionner des exemples a lire a la main.
VAGUE_PATTERNS = [
    "cela depend", "ca depend", "it depends", "dipende", "depende",
    "es kommt darauf an", "varie selon", "varies depending",
]

STEREOTYPE_PATTERNS = [
    "toujours", "jamais", "tous les", "toutes les", "les gens de",
    "always", "never", "all people", "typically", "traditionally",
    "siempre", "nunca", "todos", "tradicionalmente",
    "sempre", "mai", "tutti", "tradizionalmente",
]

# Detection de la langue de la reponse (exigence du sujet : "repond-il dans la
# bonne langue ?"). Deterministe via seed fixe.
try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0
    _LANGDETECT_OK = True
except Exception:
    _LANGDETECT_OK = False
    print("langdetect absent : answer_lang/lang_mismatch non calcules.")


def detect_lang(text):
    text = str(text).strip()
    if not _LANGDETECT_OK or len(text) < 3:
        return "unknown"
    try:
        return detect(text)
    except Exception:
        return "unknown"


def normalize_text(text):
    text = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(char for char in text if not unicodedata.combining(char))


def sentence_count(text):
    text = str(text).strip()
    if not text:
        return 0
    return max(1, len([part for part in re.split(r"[.!?]+", text) if part.strip()]))


def extract_context_country(prompt, lang=None):
    text = str(prompt)
    rx = _COMPILED_BY_LANG.get(str(lang or "").lower())
    if rx is not None:
        match = rx.search(text)
        return match.group(1).strip(" .,:;!?") if match else "unknown"
    # Langue inconnue : on tente chaque pattern dans un ordre deterministe.
    for key in _FALLBACK_ORDER:
        match = _COMPILED_BY_LANG[key].search(text)
        if match:
            return match.group(1).strip(" .,:;!?")
    return "unknown"


def mentions_context_country(answer, context_country):
    if context_country in {"", "unknown", "implicit"}:
        return False
    return normalize_text(context_country) in normalize_text(answer)


def add_text_metrics(df):
    if df.empty:
        return df.copy()
    df = df.copy()
    answer_norm = df["answer"].map(normalize_text)
    # Routage par langue : on passe la colonne "language" a l'extracteur.
    df["context_country"] = np.where(
        df["dataset_type"].eq("specific"),
        [extract_context_country(prompt, lang)
         for prompt, lang in zip(df["prompt"], df["language"])],
        "implicit",
    )
    df["answer_words"] = df["answer"].str.split().str.len().fillna(0).astype(int)
    df["sentence_count"] = df["answer"].map(sentence_count)
    df["is_empty"] = df["answer"].str.strip().eq("")
    df["too_short"] = df["answer_words"].lt(3)
    df["too_long"] = df["answer_words"].gt(40) | df["sentence_count"].gt(2)
    df["mentions_context_country"] = [
        mentions_context_country(answer, country)
        for answer, country in zip(df["answer"], df["context_country"])
    ]
    # Renommees *_keyword_hit : ce sont des detections LEXICALES, pas semantiques.
    df["vague_keyword_hit"] = answer_norm.map(lambda text: any(pattern in text for pattern in VAGUE_PATTERNS))
    df["stereotype_keyword_hit"] = answer_norm.map(lambda text: any(pattern in text for pattern in STEREOTYPE_PATTERNS))
    # Langue de la reponse + mismatch vs langue attendue.
    df["answer_lang"] = df["answer"].map(detect_lang)
    df["lang_mismatch"] = df["answer_lang"].ne(df["language"]) & df["answer_lang"].ne("unknown")
    df["instruction_issue"] = df["is_empty"] | df["too_long"] | df["mentions_context_country"]
    return df


df_all = add_text_metrics(df_all)
if not df_all.empty:
    display(df_all[[
        "run_label", "language", "id", "context_country", "answer_words",
        "sentence_count", "is_empty", "too_long", "mentions_context_country",
        "vague_keyword_hit", "stereotype_keyword_hit", "answer_lang", "lang_mismatch",
    ]].head())


In [ ]:
def summarize_metrics(df, group_cols):
    if df.empty:
        return pd.DataFrame()
    summary = (
        df.groupby(group_cols, dropna=False)
        .agg(
            n=("id", "count"),
            empty_pct=("is_empty", lambda values: values.mean() * 100),
            avg_words=("answer_words", "mean"),
            median_words=("answer_words", "median"),
            too_long_pct=("too_long", lambda values: values.mean() * 100),
            mentions_country_pct=("mentions_context_country", lambda values: values.mean() * 100),
            vague_kw_pct=("vague_keyword_hit", lambda values: values.mean() * 100),
            stereotype_kw_pct=("stereotype_keyword_hit", lambda values: values.mean() * 100),
            lang_mismatch_pct=("lang_mismatch", lambda values: values.mean() * 100),
        )
        .round(2)
        .reset_index()
    )
    return summary


run_summary = summarize_metrics(df_all, ["provider", "model", "strategy", "dataset_type"])
language_summary = summarize_metrics(df_all, ["provider", "model", "strategy", "dataset_type", "language"])

display(run_summary)
display(language_summary)

# Verification dediee de la langue de reponse (exigence du sujet).
if "lang_mismatch" in df_all.columns:
    lang_check = (
        df_all.groupby(["provider", "strategy", "dataset_type", "language"])["lang_mismatch"]
        .mean().mul(100).round(1).reset_index(name="lang_mismatch_pct")
    )
    print("Taux de reponse dans la mauvaise langue (%) :")
    display(lang_check)


In [ ]:
if not language_summary.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    plot_df = language_summary.copy()
    plot_df["condition"] = plot_df["provider"] + " / " + plot_df["strategy"] + " / " + plot_df["dataset_type"]

    sns.barplot(data=plot_df, x="language", y="avg_words", hue="condition", ax=axes[0])
    axes[0].set_title("Longueur moyenne")
    axes[0].set_ylabel("Mots")

    sns.barplot(data=plot_df, x="language", y="empty_pct", hue="condition", ax=axes[1])
    axes[1].set_title("Reponses vides")
    axes[1].set_ylabel("%")

    sns.barplot(data=plot_df, x="language", y="mentions_country_pct", hue="condition", ax=axes[2])
    axes[2].set_title("Mention du pays du prompt")
    axes[2].set_ylabel("%")

    for ax in axes:
        ax.set_xlabel("Langue")
        ax.legend_.remove()
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()

    FIG_DIR = PROJECT_ROOT / "data" / "output" / "analysis" / "figures"
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / "metrics_par_langue.png", dpi=120, bbox_inches="tight")


## 5. Embeddings semantiques

Cette partie repond a l'exigence de mesure semantique. Le cache evite de recalculer les embeddings quand les runs complets arrivent.

In [ ]:
def make_fingerprint(values):
    digest = hashlib.sha1()
    for value in values:
        digest.update(str(value).encode("utf-8", errors="ignore"))
        digest.update(b"\0")
    return digest.hexdigest()[:12]


def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def embedding_groups(df):
    group_cols = ["run_id", "provider", "model", "strategy", "dataset_type"]
    return df.groupby(group_cols, dropna=False)


def load_or_compute_embeddings(df, model_name=EMBEDDING_MODEL_NAME):
    if df.empty:
        return {}
    if importlib.util.find_spec("sentence_transformers") is None:
        print("sentence-transformers absent : analyses semantiques ignorees.")
        return {}

    from sentence_transformers import SentenceTransformer

    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    model = SentenceTransformer(model_name)
    embeddings = {}

    for keys, group in embedding_groups(df):
        answers = group["answer"].fillna("").astype(str).tolist()
        fingerprint = make_fingerprint(group["id"].tolist() + answers)
        cache_key = safe_name("__".join(map(str, keys)) + f"__{len(group)}__{fingerprint}")
        cache_path = CACHE_DIR / f"{cache_key}.npy"

        if cache_path.exists():
            array = np.load(cache_path)
            if len(array) == len(group):
                embeddings[keys] = array
                print(f"Cache charge : {cache_path.name}")
                continue

        array = model.encode(answers, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
        np.save(cache_path, array)
        embeddings[keys] = array
        print(f"Embeddings calcules : {cache_path.name}")

    return embeddings


embeddings_by_run = load_or_compute_embeddings(df_all)

In [ ]:
def attach_embeddings(df, embeddings):
    if df.empty or not embeddings:
        return pd.DataFrame()
    frames = []
    group_cols = ["run_id", "provider", "model", "strategy", "dataset_type"]
    for keys, group in df.groupby(group_cols, dropna=False):
        array = embeddings.get(keys)
        if array is None or len(array) != len(group):
            continue
        group = group.copy()
        group["embedding"] = list(array)
        frames.append(group)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def mean_pairwise_cosine(array):
    if len(array) < 2:
        return np.nan
    from sklearn.metrics.pairwise import cosine_similarity
    matrix = cosine_similarity(array)
    values = [matrix[i, j] for i, j in combinations(range(len(array)), 2)]
    return float(np.mean(values)) if values else np.nan


df_emb = attach_embeddings(df_all, embeddings_by_run)
print(f"Reponses avec embeddings : {len(df_emb)}")

## 6. Coherence specific et diversite unspecific

- `specific` : on attend une forte coherence entre langues pour la meme question et le meme contexte culturel.
- `unspecific` : on attend davantage de diversite entre langues pour la meme question.

In [ ]:
SPECIFIC_GROUP_COLS = ["provider", "model", "strategy", "question_id", "culture_id"]
UNSPECIFIC_GROUP_COLS = ["provider", "model", "strategy", "question_id"]


def compute_specific_coherence(df):
    specific = df[df["dataset_type"].eq("specific")].copy()
    if specific.empty or "embedding" not in specific.columns:
        return pd.DataFrame()
    rows = []
    for keys, group in specific.groupby(SPECIFIC_GROUP_COLS, dropna=False):
        if group["language"].nunique() < 2:
            continue
        country = "unknown"
        if "context_country" in group.columns and not group["context_country"].empty:
            modes = group["context_country"].mode()
            if not modes.empty:
                country = modes.iat[0]
        rows.append({
            "provider": keys[0],
            "model": keys[1],
            "strategy": keys[2],
            "question_id": keys[3],
            "culture_id": keys[4],
            # context_country reste informatif mais N'EST PLUS une cle de groupe.
            "context_country": country,
            "n_languages": group["language"].nunique(),
            "specific_coherence": mean_pairwise_cosine(np.stack(group["embedding"].values)),
        })
    return pd.DataFrame(rows)


def compute_unspecific_diversity(df):
    unspecific = df[df["dataset_type"].eq("unspecific")].copy()
    if unspecific.empty or "embedding" not in unspecific.columns:
        return pd.DataFrame()
    rows = []
    for keys, group in unspecific.groupby(UNSPECIFIC_GROUP_COLS, dropna=False):
        if group["language"].nunique() < 2:
            continue
        similarity = mean_pairwise_cosine(np.stack(group["embedding"].values))
        rows.append({
            "provider": keys[0],
            "model": keys[1],
            "strategy": keys[2],
            "question_id": keys[3],
            "n_languages": group["language"].nunique(),
            "avg_language_similarity": similarity,
            "unspecific_diversity": 1 - similarity,
        })
    return pd.DataFrame(rows)


specific_coherence = compute_specific_coherence(df_emb)
unspecific_diversity = compute_unspecific_diversity(df_emb)

if specific_coherence.empty:
    print("Coherence specific non calculee : embeddings ou langues alignees insuffisants.")
else:
    print(f"Cellules specific exploitables (>=2 langues) : {len(specific_coherence)}")
    print("Repartition du nombre de langues comparees par item :")
    display(specific_coherence["n_languages"].value_counts().sort_index())
    print("Coherence moyenne par condition (toutes cellules >=2 langues) :")
    display(specific_coherence.groupby(["provider", "model", "strategy"]).agg(
        n_cases=("question_id", "count"),
        avg_coherence=("specific_coherence", "mean"),
        median_coherence=("specific_coherence", "median"),
    ).round(3).reset_index())

    # ETAPE 3 : sous-ensemble STRICT (toutes les langues cible presentes).
    full = specific_coherence[specific_coherence["n_languages"].eq(len(TARGET_LANGUAGES))]
    print(f"\nSous-ensemble STRICT ({len(TARGET_LANGUAGES)} langues) : {len(full)} items")
    if not full.empty:
        display(full.groupby(["provider", "model", "strategy"]).agg(
            n_cases=("question_id", "count"),
            avg_coherence_full=("specific_coherence", "mean"),
            median_coherence_full=("specific_coherence", "median"),
        ).round(3).reset_index())

if unspecific_diversity.empty:
    print("Diversite unspecific non calculee : embeddings ou langues alignees insuffisants.")
else:
    print(f"Questions unspecific exploitables : {len(unspecific_diversity)}")
    display(unspecific_diversity.groupby(["provider", "model", "strategy"]).agg(
        n_questions=("question_id", "count"),
        avg_diversity=("unspecific_diversity", "mean"),
        median_diversity=("unspecific_diversity", "median"),
    ).round(3).reset_index())


In [ ]:
if not specific_coherence.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_df = specific_coherence.copy()
    plot_df["condition"] = plot_df["provider"] + " / " + plot_df["strategy"]
    sns.histplot(data=plot_df, x="specific_coherence", hue="condition", bins=30, kde=True, ax=ax)
    ax.set_title("Specific - coherence inter-langues")
    ax.set_xlabel("Similarite cosinus moyenne")
    plt.tight_layout()

if not unspecific_diversity.empty:
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_df = unspecific_diversity.copy()
    plot_df["condition"] = plot_df["provider"] + " / " + plot_df["strategy"]
    sns.histplot(data=plot_df, x="unspecific_diversity", hue="condition", bins=20, kde=True, ax=ax)
    ax.set_title("Unspecific - diversite inter-langues")
    ax.set_xlabel("1 - similarite cosinus moyenne")
    plt.tight_layout()

FIG_DIR = PROJECT_ROOT / "data" / "output" / "analysis" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
for _i, _fig in enumerate(map(plt.figure, plt.get_fignums())):
    pass
if plt.get_fignums():
    plt.gcf().savefig(FIG_DIR / "coherence_diversite.png", dpi=120, bbox_inches="tight")


## 7. Genericite semantique

La genericite est estimee par proximite au centroide semantique du run. Les reponses les plus proches du centroide sont candidates a une lecture qualitative.

In [ ]:
def add_semantic_genericity(df):
    if df.empty or "embedding" not in df.columns:
        return df.copy()
    from sklearn.metrics.pairwise import cosine_similarity
    frames = []
    group_cols = ["provider", "model", "strategy", "dataset_type"]
    for _, group in df.groupby(group_cols, dropna=False):
        group = group.copy()
        array = np.stack(group["embedding"].values)
        centroid = array.mean(axis=0, keepdims=True)
        similarity_to_centroid = cosine_similarity(array, centroid).ravel()
        group["semantic_specificity"] = 1 - similarity_to_centroid
        threshold = np.percentile(group["semantic_specificity"], 25)
        group["genericity_risk"] = group["semantic_specificity"].le(threshold)
        frames.append(group)
    return pd.concat(frames, ignore_index=True) if frames else df.copy()


df_analysis = add_semantic_genericity(df_emb) if not df_emb.empty else df_all.copy()

if "genericity_risk" in df_analysis.columns:
    display(df_analysis.groupby(["provider", "model", "strategy", "dataset_type"]).agg(
        n=("id", "count"),
        genericity_pct=("genericity_risk", lambda values: values.mean() * 100),
        avg_specificity=("semantic_specificity", "mean"),
    ).round(3).reset_index())

## 8. Comparaisons structurees

Les comparaisons sont faites uniquement sur des items alignes : meme dataset, meme langue, meme identifiant de question. Cela evite de comparer des sous-echantillons differents.

In [ ]:
def aligned_subset(df, mask_left, mask_right, key_cols):
    left = df.loc[mask_left].copy()
    right = df.loc[mask_right].copy()
    left_keys = set(map(tuple, left[key_cols].drop_duplicates().to_numpy()))
    right_keys = set(map(tuple, right[key_cols].drop_duplicates().to_numpy()))
    common_keys = left_keys & right_keys
    left = left[left[key_cols].apply(tuple, axis=1).isin(common_keys)]
    right = right[right[key_cols].apply(tuple, axis=1).isin(common_keys)]
    return left, right, common_keys


def compare_conditions(df, mask_left, mask_right, label_left, label_right):
    key_cols = ["dataset_type", "language", "id"]
    left, right, common_keys = aligned_subset(df, mask_left, mask_right, key_cols)
    if not common_keys:
        return pd.DataFrame()
    result = []
    for label, part in [(label_left, left), (label_right, right)]:
        summary = summarize_metrics(part, ["dataset_type"])
        summary.insert(0, "condition", label)
        summary.insert(1, "aligned_items", len(common_keys))
        result.append(summary)
    return pd.concat(result, ignore_index=True)

In [ ]:
baseline_variant_tables = []
for (provider, model, dataset_type), group in df_all.groupby(["provider", "model", "dataset_type"], dropna=False):
    strategies = sorted(set(group["strategy"]))
    if BASELINE_STRATEGY not in strategies:
        continue
    for strategy in strategies:
        if strategy == BASELINE_STRATEGY:
            continue
        table = compare_conditions(
            group,
            group["strategy"].eq(BASELINE_STRATEGY),
            group["strategy"].eq(strategy),
            BASELINE_STRATEGY,
            strategy,
        )
        if not table.empty:
            table.insert(0, "provider", provider)
            table.insert(1, "model", model)
            table.insert(2, "variant", strategy)
            baseline_variant_tables.append(table)

if baseline_variant_tables:
    baseline_variant_comparison = pd.concat(baseline_variant_tables, ignore_index=True)
    display(baseline_variant_comparison)
else:
    print("Comparaison baseline/variante en attente : aucun couple vanilla/variante aligne pour l'instant.")

In [ ]:
model_tables = []
for (strategy, dataset_type), group in df_all.groupby(["strategy", "dataset_type"], dropna=False):
    providers = sorted(group["provider"].dropna().unique())
    for left_provider, right_provider in combinations(providers, 2):
        table = compare_conditions(
            group,
            group["provider"].eq(left_provider),
            group["provider"].eq(right_provider),
            left_provider,
            right_provider,
        )
        if not table.empty:
            table.insert(0, "strategy", strategy)
            table.insert(1, "comparison", f"{left_provider} vs {right_provider}")
            model_tables.append(table)

if model_tables:
    model_comparison = pd.concat(model_tables, ignore_index=True)
    display(model_comparison)
else:
    print("Comparaison modele/modele en attente : pas d'items alignes suffisants.")

## 9. Analyse qualitative

Les categories ci-dessous servent a selectionner des exemples a lire et commenter dans le rapport. Elles ne remplacent pas une annotation humaine.

In [ ]:
def show_examples(df, mask, title, n=3, sort_col=None, ascending=True):
    subset = df.loc[mask].copy()
    print("\n" + "=" * 100)
    print(f"{title} - {len(subset)} cas")
    print("=" * 100)
    if subset.empty:
        print("Aucun exemple.")
        return
    if sort_col and sort_col in subset.columns:
        subset = subset.sort_values(sort_col, ascending=ascending)
    for _, row in subset.head(n).iterrows():
        print(f"[{row['provider']} / {row['strategy']} / {row['dataset_type']} / {row['language']}] id={row['id']}")
        print(f"Pays/contexte : {row.get('context_country', 'n/a')}")
        print("Prompt  :", str(row["prompt"])[:240])
        print("Reponse :", str(row["answer"])[:500])
        print("-" * 100)


if not df_analysis.empty:
    show_examples(df_analysis, df_analysis["is_empty"], "Reponses vides")
    show_examples(df_analysis, df_analysis["too_long"], "Non-respect de la concision")
    show_examples(df_analysis, df_analysis["mentions_context_country"], "Mention du pays/contexte malgre la consigne")
    show_examples(df_analysis, df_analysis["vague_keyword_hit"], "Reponses vagues")
    show_examples(df_analysis, df_analysis["stereotype_keyword_hit"], "Risque de stereotype ou generalisation")
    if "genericity_risk" in df_analysis.columns:
        show_examples(df_analysis, df_analysis["genericity_risk"], "Genericite semantique", sort_col="semantic_specificity")

In [ ]:
if not specific_coherence.empty:
    print("Cas specific les moins coherents entre langues :")
    display(specific_coherence.sort_values("specific_coherence").head(10))
    print("Cas specific les plus coherents entre langues :")
    display(specific_coherence.sort_values("specific_coherence", ascending=False).head(10))

if not unspecific_diversity.empty:
    print("Cas unspecific les plus variables entre langues :")
    display(unspecific_diversity.sort_values("unspecific_diversity", ascending=False).head(10))
    print("Cas unspecific les moins variables entre langues :")
    display(unspecific_diversity.sort_values("unspecific_diversity").head(10))

## 10. Discussion des raisons possibles

Points a discuter dans le rapport final :

- reponses vides : timeout, quota, erreur provider, filtrage ou probleme de parsing ;
- reponses trop longues : consigne de concision insuffisamment suivie ou modele trop bavard ;
- mention du pays : non-respect de la consigne ou tendance a expliciter le contexte culturel ;
- faible coherence en `specific` : instabilite entre langues, interpretation differente du meme contexte ;
- faible diversite en `unspecific` : reponses generiques ou uniformisation culturelle ;
- forte genericite : strategie de prudence du modele, manque d'ancrage culturel ;
- stereotypes possibles : formulations trop generales, normes culturelles presentees comme universelles.

## 11. Synthese operationnelle

Cette cellule donne une vue finale de ce qui est actuellement exploitable.

In [ ]:
def print_operational_summary():
    print("Synthese Lot D")
    print("-" * 80)
    print(f"Langues cible : {TARGET_LANGUAGES}")
    print(f"Reponses chargees : {len(df_all)}")

    if not run_catalog.empty:
        print("\nCompletude :")
        display(run_catalog.groupby(["strategy", "dataset_type", "status"]).size().reset_index(name="count"))

    print("\nDisponibilite des analyses :")
    print("- statistiques simples :", "OK" if not run_summary.empty else "NON")
    print("- coherence specific :", "OK" if not specific_coherence.empty else "EN ATTENTE")
    print("- diversite unspecific :", "OK" if not unspecific_diversity.empty else "EN ATTENTE")
    print("- baseline vs variante :", "OK" if "baseline_variant_comparison" in globals() else "EN ATTENTE")
    print("- analyse qualitative :", "OK" if not df_analysis.empty else "NON")

    print("\nLecture des resultats :")
    print("Les runs partiels peuvent verifier le pipeline d'analyse, mais les conclusions finales doivent attendre les datasets complets et des items alignes.")


print_operational_summary()

## 12. Export des artefacts (CSV / JSON / figures)

Fige les resultats dans `data/output/analysis/` pour qu'ils soient lisibles sans relancer le notebook.

In [ ]:
ANALYSIS_DIR = PROJECT_ROOT / "data" / "output" / "analysis"
FIG_DIR = ANALYSIS_DIR / "figures"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# 1) Tableaux -> CSV
if not run_summary.empty:
    run_summary.to_csv(ANALYSIS_DIR / "run_summary.csv", index=False, encoding="utf-8")
if not language_summary.empty:
    language_summary.to_csv(ANALYSIS_DIR / "language_summary.csv", index=False, encoding="utf-8")
if "lang_check" in globals() and not lang_check.empty:
    lang_check.to_csv(ANALYSIS_DIR / "lang_mismatch.csv", index=False, encoding="utf-8")
if not specific_coherence.empty:
    specific_coherence.to_csv(ANALYSIS_DIR / "specific_coherence.csv", index=False, encoding="utf-8")
if not unspecific_diversity.empty:
    unspecific_diversity.to_csv(ANALYSIS_DIR / "unspecific_diversity.csv", index=False, encoding="utf-8")
if "baseline_variant_comparison" in globals() and not baseline_variant_comparison.empty:
    baseline_variant_comparison.to_csv(ANALYSIS_DIR / "baseline_vs_variant.csv", index=False, encoding="utf-8")

# 2) Resume machine-lisible
summary = {
    "n_responses": int(len(df_all)),
    "target_languages": TARGET_LANGUAGES,
    "conditions": run_summary.to_dict(orient="records") if not run_summary.empty else [],
    "specific_coherence_by_condition": (
        specific_coherence.groupby(["provider", "model", "strategy"])["specific_coherence"]
        .mean().round(3).reset_index().to_dict(orient="records")
        if not specific_coherence.empty else []
    ),
    "unspecific_diversity_by_condition": (
        unspecific_diversity.groupby(["provider", "model", "strategy"])["unspecific_diversity"]
        .mean().round(3).reset_index().to_dict(orient="records")
        if not unspecific_diversity.empty else []
    ),
}
(ANALYSIS_DIR / "analysis_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Artefacts ecrits dans", ANALYSIS_DIR)
for path in sorted(ANALYSIS_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(ANALYSIS_DIR))
